# Spike Trainer (Colab GPU)
Two-stage plan: **Stage 1** pretrain spike-base on prose (~60M chars), **Stage 2** finetune to spike-chat on chat + flafi with replay guard.

Upload once to Drive: `BaseModelDataset.txt`, `stories.txt`, `chat_corpus_clean.txt`, `flafi-dataset.txt` into `/content/drive/MyDrive/spike/`. Then Runtime → Run all with a GPU runtime.


In [ ]:
# @title 0. Setup
!pip install torch numpy -q
import os
if not os.path.exists("setup.py"):
    !git clone https://github.com/Shimba-crypto/shimba-llm.git spike-src
    %cd spike-src
from google.colab import drive
drive.mount("/content/drive")
SRC = "/content/drive/MyDrive/spike"
print("cuda:", __import__("torch").cuda.is_available())
!ls -lh "$SRC"


In [ ]:
# @title 1. Blend datasets (fast, on Colab)
# stage-1: stories (19M) + first 40M chars of BaseModelDataset prose
# stage-2: full chat_corpus_clean + flafi thinking turns
SRC = "/content/drive/MyDrive/spike"
with open(f"{SRC}/stories.txt", encoding="utf-8", errors="replace") as f:
    stories = f.read()
with open(f"{SRC}/BaseModelDataset.txt", encoding="utf-8", errors="replace") as f:
    prose = f.read(40_000_000)
open("/content/spike-base.txt", "w", encoding="utf-8").write(stories + "\n\n" + prose)
with open(f"{SRC}/chat_corpus_clean.txt", encoding="utf-8", errors="replace") as f:
    chat = f.read()
with open(f"{SRC}/flafi-dataset.txt", encoding="utf-8") as f:
    flafi = f.read()
open("/content/spike-chat.txt", "w", encoding="utf-8").write(chat + "\n\n" + flafi)
!wc -c /content/spike-base.txt /content/spike-chat.txt



In [ ]:
# @title 2. Stage 1: pretrain spike-base (~6M, block 256, GPU)
# ~8000 iters. Watch val_loss fall; best checkpoint is kept automatically.
!python3 setup.py train --data /content/spike-base.txt --out spike-base.pth --device auto \
  --block_size 256 --n_embd 256 --n_layer 8 --n_head 4 \
  --batch_size 8 --grad_accum 4 --lr 3e-4 --max_iters 8000 --eval_interval 400 --eval_iters 40


In [ ]:
# @title 3. Stage 2: finetune to spike-chat (thinking + replay guard)
!python3 setup.py finetune --model spike-base.pth --data /content/spike-chat.txt --out spike-chat.pth --device auto \
  --thinking --replay /content/spike-base.txt --replay_prob 0.25 --forget_guard /content/spike-base.txt --forget_tol 0.10 \
  --lr 1e-4 --max_iters 2000 --batch_size 8 --grad_accum 4 --eval_interval 200 --eval_iters 40


In [ ]:
# @title 4. Benchmark vs bom-base (upload yours to Drive spike/ first, optional)
import os
extra = ""
if os.path.exists("/content/drive/MyDrive/spike/bom-base_final.pth"):
    extra = "--compare /content/drive/MyDrive/spike/bom-base_final.pth"
!python3 benchmark.py --model spike-chat.pth $extra 2>&1 | tail -n 30


In [ ]:
# @title 5. Greedy smoke test (must be coherent at temp 0)
!python3 setup.py generate --model spike-chat.pth --prompt "User: hi\n" --max_tokens 60 --temperature 0.0 --device auto
!python3 setup.py generate --model spike-chat.pth --prompt "User: what is 2 plus 2\n" --max_tokens 80 --temperature 0.0 --device auto


In [ ]:
# @title 6. Save back to Drive
!cp spike-base.pth spike-chat.pth spike-base_tokenizer.json spike-chat_tokenizer.json /content/drive/MyDrive/spike/ 2>/dev/null; ls -lh /content/drive/MyDrive/spike/*.pth
